# Lab 08 · An MCP server with all three primitives
**~30 minutes · costs nothing · Domain 8 MCP Development**

No API calls. You write a server, connect it to Claude Desktop or Claude Code,
and see the three primitives appear. Knowing **who controls each** is the
testable distinction:

| Primitive | Controlled by | Purpose |
|---|---|---|
| Tools | **Model** | Actions it can invoke |
| Resources | **Application** | Read-only data pulled into context |
| Prompts | **User** | Reusable templates the user invokes |

In [ ]:
%pip install -q "mcp[cli]"
print("installed")

## Write the server

In [ ]:
server = '''from mcp.server.fastmcp import FastMCP

mcp = FastMCP("warehouse")

STOCK = {"WID-1": 42, "WID-2": 0, "WID-3": 118}

@mcp.tool()
def check_stock(sku: str) -> int:
    """Return the current stock level for a SKU. Use when asked whether an
    item is available or how many remain. Do NOT use to place orders."""
    return STOCK.get(sku, -1)

@mcp.resource("catalog://all")
def catalog() -> str:
    """The full SKU catalogue as read-only reference data."""
    return "\\n".join(f"{k}: {v} units" for k, v in STOCK.items())

@mcp.prompt()
def restock_review(sku: str) -> str:
    """Template for a restocking decision."""
    return f"Review stock for {sku} and recommend whether to reorder."

if __name__ == "__main__":
    mcp.run()          # stdio: the client spawns this process
'''
open("warehouse_server.py","w").write(server)
print("wrote warehouse_server.py")

## Sanity check it starts

stdio servers speak JSON-RPC on stdin/stdout, so running it bare just blocks.
That is correct behaviour, not a hang.

In [ ]:
import subprocess, sys, time
p = subprocess.Popen([sys.executable,"warehouse_server.py"],
                     stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(2)
print("still running (expected):", p.poll() is None)
p.terminate()
err = p.stderr.read().decode()[:400]
print("stderr:", err if err else "(clean)")

## Connect it

**Claude Code**
```
claude mcp add warehouse -- python /full/path/to/warehouse_server.py
```

**Claude Desktop** — add to `claude_desktop_config.json`:
```json
{"mcpServers":{"warehouse":{"command":"python",
  "args":["/full/path/to/warehouse_server.py"]}}}
```

Restart, then ask *"how many WID-3 do we have?"* and watch the tool get called.

### The transport heuristic
The config above gives a **command to run** → that is **stdio**, local, no
network exposure. If a server's docs give you a **URL** instead, that is
**Streamable HTTP**, which is the current remote standard and needs auth.
The older HTTP+SSE transport is deprecated.

---
### Checkpoint
- Name the three primitives and who controls each
- Which transport for a local process? Which for remote?
- When is an MCP server the right answer instead of a custom tool?